# Finish Russian construction scoring on a Colab GPU

This notebook scores the exported unfinished RNC targets with the same ruRoberta-large checkpoint and float32 word-scoring method. It does not collect new corpus sentences and needs no RNC token.

**Before starting:** select **Runtime → Change runtime type → GPU**. Obtain `colab_transfer.zip` from the project workspace. The bundle contains corpus sentences: keep it in your own Drive, not a public GitHub repository.

Run cells in order. Keep the existing CPU run alive during setup. After the benchmark, share its printed estimate before deciding whether to complete the GPU run. Outputs are saved in your Drive and can be resumed with the same bundle and environment.


## 1. Check the GPU and install the small scoring dependencies

In [ ]:
import subprocess, sys
subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.6", "pymorphy3==2.0.6",
                "pymorphy3-dicts-ru==2.4.417150.4580142"], check=True)
import torch
assert torch.cuda.is_available(), "Choose a GPU runtime first."
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
# Keep Colab's CUDA-enabled PyTorch. Do not install the CPU-only project lockfile.


## 2. Connect your Drive and upload the transfer bundle once

In [ ]:
from google.colab import drive, files
from pathlib import Path
import hashlib, json, shutil, zipfile

drive.mount("/content/drive")
storage = Path("/content/drive/MyDrive/ru-constructions-colab")
storage.mkdir(parents=True, exist_ok=True)
bundle = storage / "colab_transfer.zip"
if not bundle.exists():
    uploaded = files.upload()  # Select the colab_transfer.zip supplied with this notebook.
    assert len(uploaded) == 1, "Upload exactly one transfer zip."
    filename, content = next(iter(uploaded.items()))
    assert filename.endswith(".zip"), "Select the transfer ZIP, not the notebook."
    bundle.write_bytes(content)
    del uploaded, content
else:
    print("Reusing your saved transfer bundle:", bundle)

workspace = Path("/content/ru-rnc-gpu")
workspace.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    for name in archive.namelist():
        assert (workspace / name).resolve().is_relative_to(workspace.resolve()), "Unsafe archive path"
    archive.extractall(workspace)
metadata = json.loads((workspace / "transfer.json").read_text())
transfer_id = hashlib.sha256((workspace / "transfer.json").read_bytes()).hexdigest()[:12]
output_dir = storage / ("results-" + transfer_id)
print("Unfinished targets in this snapshot:", metadata["pending_targets"])
print("Results will be saved to:", output_dir)


## 3. Download and verify the exact model used by the CPU run

In [ ]:
from huggingface_hub import hf_hub_download
model_dir = workspace / "data/raw/model"
model_dir.mkdir(parents=True, exist_ok=True)
for entry in metadata["model_files"]:
    filename = Path(entry["path"]).name
    hf_hub_download("ai-forever/ruRoberta-large", filename=filename,
                    revision=metadata["model_revision"], local_dir=model_dir, token=False)
    with (model_dir / filename).open("rb") as source:
        actual = hashlib.file_digest(source, "sha256").hexdigest()
    assert actual == entry["sha256"], f"Model checksum mismatch: {filename}"
print("Pinned model files verified.")


## 4. Validate CPU/GPU agreement and benchmark remaining sentence lengths

This checks saved CPU predictions and additional long unfinished contexts. It stops on mismatched recovery decisions or probabilities outside the stated tolerance. The estimate covers scoring only and may vary with GPU load and Drive performance.

**Share the printed `estimated_scoring_hours` and validation output with the assistant.**

In [ ]:
command = [sys.executable, str(workspace / "scripts/score_colab.py"),
           "--output-dir", str(output_dir)]
subprocess.run(command + ["--benchmark-only"], cwd=workspace, check=True)
print((output_dir / "gpu_validation.json").read_text())


## 5. Score the remaining targets

Run this after the checks pass. Predictions are flushed to Drive after each batch. If the runtime disconnects, reconnect a GPU and rerun the notebook using the saved bundle. The scorer skips completed targets. Keep the same software versions; if it detects an environment change, consult the assistant before changing output folders or discarding results.


In [ ]:
subprocess.run(command, cwd=workspace, check=True)
print((output_dir / "gpu_completion.json").read_text())


## 6. Download the completed results

Return this ZIP to the assistant, or place it in the shared project workspace and provide its path. The local CPU scorer must be stopped before merging. The importer keeps existing CPU records and checks overlaps before adding GPU results. Do not overwrite the local score file manually.


In [ ]:
completion = json.loads((output_dir / "gpu_completion.json").read_text())
assert completion["complete"]
result_zip = storage / ("gpu-results-" + transfer_id + ".zip")
with zipfile.ZipFile(result_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in ["gpu_scores.jsonl", "gpu_metadata.json", "gpu_validation.json", "gpu_completion.json"]:
        archive.write(output_dir / name, arcname=name)
print("A copy remains in Drive:", result_zip)
files.download(str(result_zip))
